# DigitalAutoresearchRunner — knob sweep via OpenCode backend

Companion to `../claude_design_slides/digital.html`, Part 4. Demonstrates
`DigitalAutoresearchRunner` with `backend="opencode"` on the 4-bit counter.

The runner accepts **four** backends — `adk`, `cc_cli`, `litellm`,
`opencode`. We default to `opencode` here because multi-provider is an
economic feature for a loop that burns five LibreLane runs per invocation.

**Default mode is DRY-RUN.** Real run gated by `RUN_REAL=True` (needs docker
+ `hpretl/iic-osic-tools` + opencode CLI + provider key).

**Prerequisites** — see `./README.md`. For this notebook specifically:

- `npm install -g opencode-ai` — puts `opencode` on PATH.
- `OPENROUTER_API_KEY` (or another provider key matching `OPENCODE_MODEL`).
- Docker + `hpretl/iic-osic-tools:next` image (real runs only).

## Step 0 — Editable install

In [ ]:
import os, sys, shutil, subprocess, json
from pathlib import Path

RUN_PIP_INSTALL   = False
RUN_DRY           = True
RUN_REAL          = False

BACKEND           = "opencode"                                    # adk | cc_cli | litellm | opencode
OPENCODE_MODEL    = "openrouter/google/gemini-3-flash-preview"    # or zai-coding-plan/glm-5.1, etc.
OPENCODE_CLI_PATH = "opencode"
BUDGET            = 5

REPO_ROOT = Path.cwd().resolve().parents[2]
WORK_DIR  = Path.cwd() / "digital_autoresearch_results"
PROJ_DIR  = WORK_DIR / "counter_project"

print(f"Repo root              : {REPO_ROOT}")
print(f"venv                   : {os.environ.get('VIRTUAL_ENV', 'UNSET')}")

if RUN_PIP_INSTALL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], check=True)
else:
    print("[rehearse] RUN_PIP_INSTALL=False — skipping.")

## Step 1 — Docker + opencode CLI + provider key check

In [ ]:
print(f"docker on PATH          : {shutil.which('docker') or 'MISSING'}")
print(f"opencode on PATH        : {shutil.which(OPENCODE_CLI_PATH) or 'MISSING -- npm i -g opencode-ai'}")
for key in ("OPENROUTER_API_KEY", "GOOGLE_API_KEY", "ZAI_API_KEY", "ANTHROPIC_API_KEY"):
    print(f"{key:<22}: {'set' if os.environ.get(key) else 'unset'}")
print(f"\nselected backend        : {BACKEND}")
if BACKEND == "opencode":
    print(f"selected opencode model : {OPENCODE_MODEL}")

## Step 2 — Stage `counter.v` + `config.yaml` (same as sibling)

In [ ]:
COUNTER_V = '''module counter (
    input  wire       clk,
    input  wire       rst,
    output wire [3:0] q
);
    reg [3:0] cnt;
    always @(posedge clk) begin
        if (rst) cnt <= 4'b0;
        else     cnt <= cnt + 4'b1;
    end
    assign q = cnt;
endmodule
'''

CONFIG_YAML = '''meta:
  version: 3
  flow: Classic
  substituting_steps:
    Magic.StreamOut: null
    KLayout.XOR: null
    KLayout.DRC: null            # gf180mcuD ships no KLAYOUT_DRC_RUNSET; the step crashes
                                 # with "Unable to open file: .../None" when librelane is
                                 # invoked without --manual-pdk (the typical agent path).
                                 # Magic DRC at 0 is the authoritative DRC for this PDK.

RUN_MAGIC_STREAMOUT: false
RUN_KLAYOUT_XOR: false
RUN_KLAYOUT_DRC: false

DESIGN_NAME: counter
VERILOG_FILES:
  - dir::counter.v
CLOCK_PORT: clk
CLOCK_PERIOD: 50

FP_SIZING: absolute
DIE_AREA: [0.0, 0.0, 300.0, 300.0]

VDD_NETS:
  - VDD
GND_NETS:
  - VSS

PRIMARY_GDSII_STREAMOUT_TOOL: klayout
DIODE_ON_PORTS: in
RT_MAX_LAYER: Metal4
PDN_MULTILAYER: false
PDN_VWIDTH: 5
PDN_HWIDTH: 5
PDN_VSPACING: 1
PDN_HSPACING: 1
PDN_VOFFSET: 10
PDN_HOFFSET: 10
'''

PROJ_DIR.mkdir(parents=True, exist_ok=True)
(PROJ_DIR / "counter.v").write_text(COUNTER_V)
(PROJ_DIR / "config.yaml").write_text(CONFIG_YAML)
for p in PROJ_DIR.iterdir():
    print(f"wrote {p}")

## Step 3 — Construct `DigitalAutoresearchRunner` and dry-run

Wraps the config with `GenericDesign` and constructs the greedy runner.
No real LLM call, no Docker touch — prints what it would do.

In [ ]:
import asyncio
from eda_agents.core.designs.generic import GenericDesign
from eda_agents.agents.digital_autoresearch import DigitalAutoresearchRunner

design = GenericDesign(
    config_path=str(PROJ_DIR / "config.yaml"),
    pdk_root=os.environ.get("PDK_ROOT") or None,
    pdk_config="gf180mcu",
)
kwargs = dict(design=design, backend=BACKEND, budget=BUDGET)
if BACKEND == "opencode":
    kwargs.update(opencode_cli_path=OPENCODE_CLI_PATH, opencode_model=OPENCODE_MODEL)
runner = DigitalAutoresearchRunner(**kwargs)

if RUN_DRY:
    print(f"design  : {design.project_name()}")
    print(f"specs   : {design.specs_description()}")
    print(f"FoM     : {design.fom_description()}")
    print(f"backend : {BACKEND}")
    if BACKEND == 'opencode':
        print(f"model   : {OPENCODE_MODEL}")
    print(f"budget  : {BUDGET} LibreLane evaluations")
    print("knobs   : density x clock period x PDN pitch (discrete)")
else:
    print('[rehearse] RUN_DRY=False — skipping.')

## Step 4 — Real run (gated)

Only flip `RUN_REAL=True` once docker + image + opencode + provider key are
all green. Expected wall time: ~5 × (5–10 min per LibreLane run on the
counter). LLM cost at Gemini Flash prices: well under $0.50 for a full
sweep.

The runner writes `program.md` + `results.tsv` under the work dir. Crashed
run? Re-run — it reads its own history back.

In [ ]:
async def _real():
    return await runner.run(WORK_DIR)

if RUN_REAL:
    res = asyncio.get_event_loop().run_until_complete(_real())
    print(json.dumps({k: v for k, v in res.__dict__.items() if not k.startswith('_')},
                     indent=2, default=str))
else:
    print('[rehearse] RUN_REAL=False — skipping.')
    print(f'When enabled: artifacts land under {WORK_DIR}')

## Step 5 — Read the trace

In [ ]:
for fname in ('program.md', 'results.tsv'):
    p = WORK_DIR / fname
    if p.exists():
        print(f'--- {p} ---')
        print(p.read_text()[:2000])
        print()
    else:
        print(f'{p} not yet written.')

## Swap backends

Change `BACKEND` in Step 0 and re-run:

- `BACKEND = "adk"` — Google ADK runner, needs `GOOGLE_API_KEY` or OpenRouter.
- `BACKEND = "cc_cli"` — `claude --print` via `ClaudeCodeHarness`, needs the `claude` CLI + Anthropic subscription.
- `BACKEND = "opencode"` — `opencode run --format json`, needs `opencode` CLI + any provider key.
- `BACKEND = "litellm"` — direct API call, no CLI; fastest cheapest exploration.

## Cleanup

```bash
rm -rf digital_autoresearch_results/
```

## Next

- End-to-end flow without greedy search: `./agents_rtl2gds_counter.ipynb`.
- Analog chain (recommender → sizing → autoresearch → corners): `./agents_analog_topology_to_sizing.ipynb`.
- Slide deck: `../claude_design_slides/digital.html`.